In [1]:
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)

import sys

sys.path.append("../src/reach_skill")

import argparse
import os

os.environ["MUJOCO_GL"] = "egl"

import time

import gymnasium as gym
import numpy as np
import torch

from arm_reach_env import UR3eReachEnv
from train_ppo_ur3e import ActorCritic

In [ ]:
class ExpArgs:
    device="cpu"
    model_path="/workspace/models/ppo_ur3e_cleanrl_ppo_ur3e_seed_42_1781698294_best.pth"
    seed=42
    episodes=1
    delay=0.01
    render_mode="human"

args = ExpArgs()

In [3]:
device = torch.device(args.device if torch.cuda.is_available() or args.device == "cpu" else "cpu")
print(f"[enjoy] loading model from {args.model_path} on {device}")

env = UR3eReachEnv(render_mode=args.render_mode)
env = gym.wrappers.TimeLimit(env, max_episode_steps=500)
print(f"[enjoy] created environment with render_mode={args.render_mode}")
obs, _ = env.reset(seed=args.seed)
print(f"[enjoy] environment reset with seed={args.seed}")



[enjoy] loading model from /workspace/models/ppo_ur3e_cleanrl_ppo_ur3e_seed_42_1781698294_best.pth on cpu
[enjoy] created environment with render_mode=human
[enjoy] environment reset with seed=42


In [4]:

obs_space = env.observation_space
act_space = env.action_space
obs_dim = int(np.prod(obs_space.shape))
act_dim = int(np.prod(act_space.shape))

model = ActorCritic(obs_dim, act_dim).to(device)
model.load_state_dict(torch.load(args.model_path, map_location=device))
model.eval()
print("[enjoy] model loaded and ready")

[enjoy] model loaded and ready


In [5]:
control_signal = []

for episode in range(1, args.episodes + 1):
    print(f"[enjoy] starting episode {episode}/{args.episodes}")
    obs, _ = env.reset(seed=args.seed + episode)
    obs = torch.tensor(obs, dtype=torch.float32, device=device)
    done = False
    truncated = False
    episode_reward = 0.0
    step = 0

    while not (done or truncated):
        with torch.no_grad():
            mean, _ = model.forward(obs)
            action = mean

        action = action.cpu().numpy()

        control_signal.append(action.tolist())
        # print("obs: ", obs)
        # print("Action: ", action)
        action = np.clip(action, act_space.low, act_space.high)
        obs, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward
        step += 1

        if args.render_mode == "human":
            env.render()
            time.sleep(args.delay)

        obs = torch.tensor(obs, dtype=torch.float32, device=device)
        done = terminated

        if step % 50 == 0:
            print(f"[enjoy] episode={episode}, step={step}, reward={episode_reward:.2f}")

    print(f"[enjoy] finished episode {episode}/{args.episodes}: reward={episode_reward:.2f}, steps={step}")



[enjoy] starting episode 1/5


[enjoy] episode=1, step=50, reward=-39.56
[enjoy] episode=1, step=100, reward=355.71
[enjoy] episode=1, step=150, reward=360.61
[enjoy] episode=1, step=200, reward=355.51
[enjoy] episode=1, step=250, reward=350.41
[enjoy] episode=1, step=300, reward=345.30
[enjoy] episode=1, step=350, reward=340.20
[enjoy] episode=1, step=400, reward=335.10
[enjoy] episode=1, step=450, reward=329.99
[enjoy] episode=1, step=500, reward=324.89
[enjoy] finished episode 1/5: reward=324.89, steps=500
[enjoy] starting episode 2/5
[enjoy] episode=2, step=50, reward=79.27
[enjoy] episode=2, step=100, reward=575.02
[enjoy] episode=2, step=150, reward=1070.73
[enjoy] episode=2, step=200, reward=1566.19
[enjoy] episode=2, step=250, reward=1781.18
[enjoy] episode=2, step=300, reward=1776.13
[enjoy] episode=2, step=350, reward=1771.06
[enjoy] episode=2, step=400, reward=1766.00
[enjoy] episode=2, step=450, reward=1760.94
[enjoy] episode=2, step=500, reward=1755.88
[enjoy] finished episode 2/5: reward=1755.88, steps

In [6]:
env.close()

In [7]:
import pandas as pd


a = np.array(control_signal)
pd.DataFrame(a.tolist()).diff().describe()

,0,1,2,3,4,5,6
count,2.499000e+03,2.499000e+03,2.499000e+03,2499.000000,2.499000e+03,2.499000e+03,2.499000e+03
mean,-1.315730e-03,9.412047e-04,-2.406750e-04,0.000042,1.096673e-04,9.027976e-05,-4.010313e-04
std,1.166354e-01,5.785545e-02,2.953044e-02,0.051673,5.166138e-02,1.314439e-02,3.580959e-02
min,-3.530492e+00,-2.605439e+00,-6.367836e-01,-0.334792,-1.987128e+00,-3.856335e-01,-1.386665e+00
25%,-5.733967e-05,-4.515052e-05,-3.267825e-05,-0.000194,-6.914139e-06,-9.402633e-05,-2.861023e-06
50%,-2.384186e-07,-5.960464e-08,-8.940697e-08,0.000000,-2.384186e-07,-1.788139e-07,1.192093e-07
75%,1.174212e-05,4.261732e-06,3.364682e-05,0.000012,6.556511e-06,6.556511e-07,4.506111e-05
max,2.966456e+00,2.802027e-01,1.059518e+00,1.921521,1.367269e-01,2.967370e-01,3.363006e-01


In [8]:
import rclpy
from rclpy.node import Node
from std_msgs.msg import Float64MultiArray
from sensor_msgs.msg import JointState